In [ ]:
import json, os, re
import pandas as pd
import gender_guesser.detector as gg

detektor = gg.Detector()

WEIBLICH = r"\b(she|her|hers|herself|woman|women|mother|mom|mum|grandmother|daughter|wife|sister|girl|lady|ladies|businesswoman|chairwoman|congresswoman|spokeswoman|mrs|ms)\b"
MAENNLICH = r"\b(he|him|his|himself|man|men|father|dad|grandfather|son|husband|brother|boy|gentleman|businessman|chairman|congressman|spokesman|mr)\b"
# Vornamen, die fuer Maenner UND Frauen ueblich sind -> immer "unbekannt"
# (bei der Handpruefung fiel "Alex" auf, der Rest ist zur Sicherheit ergaenzt)
UNEINDEUTIG = {"Alex", "Sam", "Jordan", "Taylor", "Casey", "Morgan", "Riley",
               "Jamie", "Chris", "Robin", "Kim", "Charlie", "Avery", "Quinn",
               "Jesse", "Drew", "Jordan", "Ariel", "Dana", "Jean"}
def geschlecht(text):
    # 1. Hinweis: geschlechtsbezogene Woerter (she, mother, his ...)
    w = re.search(WEIBLICH, text, re.IGNORECASE) is not None
    m = re.search(MAENNLICH, text, re.IGNORECASE) is not None
    if w and not m:
        return "weiblich"
    if m and not w:
        return "maennlich"
    if w and m:
        return "gemischt"
    # 2. Hinweis: Vorname am Anfang der Beschreibung (z. B. "Fatima Mohammed is ...")
    treffer = re.match(r"^(?:Dr\.|Professor|Prof\.)?\s*([A-Z][a-z]+) [A-Z][a-z]+", text)
    if treffer and treffer.group(1) not in UNEINDEUTIG:
        ergebnis = detektor.get_gender(treffer.group(1))
        if ergebnis == "female":
            return "weiblich"
        if ergebnis == "male":
            return "maennlich"
    return "unbekannt"

# kurzer Test
print(geschlecht("A concerned mother who has lost a child."))
print(geschlecht("Rashid Hassan is a small business owner."))
print(geschlecht("A zoologist researching ecodiversity hotspots."))

In [ ]:
# Ordner mit den Debatten von Frau Moldovan-Mauer (GPT-OSS-120B)
ordner = "../data/toxic_and_baseline_random_oss"

zeilen = []  # hier sammeln wir pro Datei eine Zeile fuer die Tabelle

for name in os.listdir(ordner):
    # nur JSON-Dateien beachten (z. B. versteckte Ordner ueberspringen)
    if not name.endswith(".json"):
        continue

    # Infos aus dem Dateinamen herausziehen:
    # m[1] = Diskussionsnummer, m[3] = Themennummer, m[5] = Toxizitaetsstufe
    m = re.match(r"discussion_(\d+)_round_(\d+)_for_topic_(\d+)_(\w+?)_toxic_level_(\w+)\.json", name)
    if m is None:
        continue  # Datei mit unerwartetem Namen ueberspringen

    # Datei oeffnen und Inhalt als Python-Woerterbuch laden
    with open(os.path.join(ordner, name), encoding="utf-8") as f:
        g = json.load(f)

    toxizitaet = m[5]

    # Persona-Beschreibungen der beiden Agenten
    pro = g["active_agents"]["pro"][0]["description"]
    con = g["active_agents"]["con"][0]["description"]

    # eine Zeile fuer die Tabelle anlegen
    zeilen.append({
        "diskussion": int(m[1]),
        "thema_nr": int(m[3]),
        "toxizitaet": toxizitaet,
        "runden": g["nrounds"],
        "einigung": g["in_alignment"],
        # bei "no" gibt es keinen toxischen Agenten
        "toxische_seite": g["pro_or_con_toxic"] if toxizitaet != "no" else "keine",
        # Geschlecht mit unserer Funktion aus Zelle 1 bestimmen
        "geschlecht_pro": geschlecht(pro),
        "geschlecht_con": geschlecht(con),
        "beschreibung_pro": pro,
        "beschreibung_con": con,
    })

# alle Zeilen zu einer Tabelle zusammenbauen
alle = pd.DataFrame(zeilen)

# pro Debatte gibt es mehrere Dateien (Zwischenstand nach jeder Runde)
# -> nur die Datei mit den meisten Runden behalten (= Endergebnis)
debatten = alle.sort_values("runden").groupby(["diskussion", "thema_nr", "toxizitaet"]).tail(1).copy()

print("Echte Debatten:", len(debatten))

In [ ]:
# --- Geschlecht des toxischen und des neutralen Agenten bestimmen ---

def geschlecht_toxisch(zeile):
    # Welcher Agent ist toxisch? Dessen Geschlecht zurueckgeben
    if zeile["toxische_seite"] == "pro":
        return zeile["geschlecht_pro"]
    if zeile["toxische_seite"] == "con":
        return zeile["geschlecht_con"]
    return "-"  # bei "no" gibt es keinen toxischen Agenten

def geschlecht_neutral(zeile):
    # Der jeweils andere Agent ist der neutrale
    if zeile["toxische_seite"] == "pro":
        return zeile["geschlecht_con"]
    if zeile["toxische_seite"] == "con":
        return zeile["geschlecht_pro"]
    return "-"

# neue Spalten in der Tabelle anlegen (axis=1 = Funktion pro Zeile anwenden)
debatten["geschlecht_toxisch"] = debatten.apply(geschlecht_toxisch, axis=1)
debatten["geschlecht_neutral"] = debatten.apply(geschlecht_neutral, axis=1)

# nur die Debatten mit Toxizitaet (mild, moderate, heavy)
toxisch = debatten[debatten["toxizitaet"] != "no"]

# 1) Geschlecht ALLER Agenten (pro und con zusammen)
print("1) Alle Agenten:")
print(pd.concat([debatten["geschlecht_pro"], debatten["geschlecht_con"]]).value_counts())
print()

# 2) Geschlecht des TOXISCHEN Agenten (wichtig fuer H1 und H2)
print("2) Toxischer Agent (nur toxische Debatten):")
print(toxisch["geschlecht_toxisch"].value_counts())
print()

# 3) Kreuztabelle: Geschlecht toxisch (Zeilen) x Geschlecht neutral (Spalten)
print("3) Toxisch (Zeilen) x Neutral (Spalten):")
print(pd.crosstab(toxisch["geschlecht_toxisch"], toxisch["geschlecht_neutral"]))

   ## Ergebnis Geschlechtserkennung (09.11.2026)

   Methode: geschlechtsbezogene Woerter (she, mother, his, ...) + Vornamen (Paket gender-guesser);
   uneindeutige Vornamen (Alex, Jesse, Sam, ...) zaehlen als "unbekannt"

   Validierung: Handpruefung von 50 zufaelligen Personas (25 w, 25 m) -> 46/50 korrekt (92 %);
   alle 4 Fehler durch uneindeutige Vornamen (3x Alex, 1x Jesse) -> danach behoben

   - Alle Agenten (7.996): weiblich 1.238, maennlich 523, unbekannt 6.220, gemischt 15
   - Toxischer Agent (2.998 toxische Debatten): weiblich 454, maennlich 201, unbekannt 2.338
   - Kombinationen toxisch x neutral (beide bekannt): m-m 11, m-w 16, w-m 31, w-w 53 -> zu wenige fuer 2x2-Design
   - Nebenbefund: KI erzeugt ca. 2,4-mal so viele weibliche wie maennliche Personas

   Grenze: Thema als moegliche Stoervariable kontrollieren

In [ ]:
# --- Stichprobe fuer die manuelle Pruefung ---

# alle Personas (pro und con) in EINE lange Liste bringen
personas = pd.concat([
    debatten[["beschreibung_pro", "geschlecht_pro"]].rename(columns={"beschreibung_pro": "beschreibung", "geschlecht_pro": "automatisch"}),
    debatten[["beschreibung_con", "geschlecht_con"]].rename(columns={"beschreibung_con": "beschreibung", "geschlecht_con": "automatisch"}),
])

# jede Persona nur einmal (dieselbe Persona kommt in mehreren Debatten vor)
personas = personas.drop_duplicates(subset="beschreibung")

# je 25 zufaellige weibliche und maennliche Personas ziehen
# random_state=42 -> bei jedem Ausfuehren dieselbe Stichprobe (reproduzierbar)
stichprobe = pd.concat([
    personas[personas["automatisch"] == "weiblich"].sample(25, random_state=42),
    personas[personas["automatisch"] == "maennlich"].sample(25, random_state=42),
])

# leere Spalte, in die du "ja" oder "nein" eintraegst
stichprobe["korrekt"] = ""

# als CSV-Datei speichern (oeffnet sich in Excel)
stichprobe.to_csv("../data/handpruefung.csv", index=False, encoding="utf-8-sig", sep=";")
print("Gespeichert:", len(stichprobe), "Personas")